<a href="https://colab.research.google.com/github/JANIKSHA-M/Campus-Connect-Platform/blob/main/CAMPUS__CONNECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [102]:
!pip install Flask-SQLAlchemy Flask-Login

In [104]:
import os
from datetime import datetime
from functools import wraps

from flask import(
    Flask, render_template,request,redirect,url_for,flash,abort
)
from flask_sqlalchemy import SQLAlchemy
from flask_login import(
    LoginManager, UserMixin,login_user,logout_user,login_required,current_user
)
from werkzeug.security import(
  generate_password_hash,check_password_hash
)

app = Flask(__name__)

app.config["SECRET_KEY"] = os.getenv(
    "SECRET_KEY","change-this-secret-key"
)

app.config["SQLALCHEMY_DATABASE_URI"] = os.getenv(
    "DATABASE_URL","sqlite:///campus_connect.db"
).replace("postgres://","postgresql://")

app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False

db = SQLAlchemy(app)

login_manager = LoginManager(app)
login_manager.login_view = "login"

In [105]:
db.metadata.clear()

class User(UserMixin, db.Model):
  id = db.Column(db.Integer, primary_key=True)
  name = db.Column(db.String(120),nullable=False)
  email = db.Column(db.String(180),unique=True, nullable=False)
  regno = db.Column(db.String(60), unique=True)
  department = db.Column(db.String(100))
  role = db.Column(db.String(40), default="student", nullable=False)
  password_hash = db.Column(db.String(255), nullable=False)
  active = db.Column(db.Boolean, default=True)

  def set_password(self, password):
    self.password_hash = generate_password_hash(password)

  def check_password(self,password):
    return check_password_hash(
        self.password_hash,password
  )

class Attendance(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    student_id = db.Column(
      db.Integer,
      db.ForeignKey("user.id"),
      nullable=False
  )
    subject = db.Column(db.String(120), nullable=False)
    class_date = db.Column(db.Date, nullable=False)
    status = db.Column(db.String(10), nullable=False)

class Mark(db.Model):
   id = db.Column(db.Integer, primary_key=True)
   student_id = db.Column(
   db.Integer,db.ForeignKey("user.id"), nullable=False)
   subject = db.Column(db.String(120), nullable=False)
   assessment = db.Column(db.String(100))
   score = db.Column(db.Float)
   maximum = db.Column(db.Float,default=100)

class Announcement(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    title = db.Column(db.String(180))
    body = db.Column(db.Text)
    author_id = db.Column(db.Integer,db.ForeignKey("user.id"))
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

class Event(db.Model):
      id = db.Column(db.Integer, primary_key=True)
      title = db.Column(db.String(180))
      details = db.Column(db.Text)
      event_date = db.Column(db.Date)
      creator_id = db.Column(db.Integer, db.ForeignKey("user.id"))

In [130]:
@login_manager.user_loader
def load_user(user_id):
  return db.session.get(User,int(user_id))

def allow(*roles):
  def decorator(function):
    @wraps(function)
    def wrapper(*args,**kwargs):
      if not current_user.is_authenticated:
         return login_manager.unauthorized()

      if current_user.role not in roles:
         abort(403)

      return function(*args,**kwargs)

    return wrapper

  return decorator

In [110]:
@app.route("/")
def home():
  return render_template("home.html")

In [109]:
@app.route("/register", methods=["GET", "POST"])
def register():

  if request.method =="POST":
    name = request.form["name"].strip()
    email = request.form["email"].strip().lower()
    password = request.form["password"]

    if len(password) < 8:
      flash("Password must contain at least8 characters.")
      return redirect(url_for("register"))

    if User.query.filter_by(email=email).first():
      flash("Email already registered.")
      return redirect(url_for("register"))

      user = User(
          name = name,
          email = email,
          regno = request.form.get("regno"),
          role="student"
      )

      user.set_password(password)

      db.session.add(user)
      db.session.commit()

      flash("Account created successfully.")
      return redirect(url_for("login"))

      return render_template("auth.html", mode="register")



In [111]:
@app.route("/login",methods=["GET","POST"])
def loginn():

  if request.method == "POST":
    email = request.form["email"].strip().lower()
    password= request.form["password"]

    user = User.query.filter-by(
        email=email,
        active=True
    ).first()

    if user and user.check_password(password):
      login_user(user)
      return redirect(url_for("dashboard"))

      flash("Invaild login details.")

      return render-template("auth.html", mode="login")


In [112]:
@app.route("/logout")
@login_required
def logout():

  logout_user()

  return redirect(url_for("home"))


In [113]:
@app.route("/dashboard")
@login_required
def dashboard():

  if current_user.role =="student":

    attendance_records = Attendance.query.filter-by(
        student_id=current_user.id
    ).all()

    mark_records = Mark.query.filter_by(
        student_id=current_user.id
    ).all()

    mark_records = Mark.query.filter_by(
        student_id=current_user.id
    ).all()

  else:
    attendance_records = None
    mark_records = None

    total = len(attendance_records)

    present = sum(
        record.status =="Present"
        for record in attendance_records
    )

    percentage = (
        round(present*100/total,1)
        if total else 0
    )

    announcements = Announcement.query.order_by(
        Announcement.created_at.desc()
    ).limit(8).all()

    events = Event.query.order_by(
        Event.event_date
    ).limit(8).all()

    return render_template(
        "dashboard.html",
        att=attendance_records,
        marks=mark_records,
        pct=percentage,
        ann=announcements,
        events=events
    )




In [115]:
@app.route("/attendance", methods=["GET", "POST"])
@login_required
def attendance():

  if request.method=="POST":

   if current_user.role not in(
      "faculty", "main_admin", "academic_admin"
  ):

    abort(403)
  record =  Attendance(
      student_id=int(request.form["student-id"]),
      subject=request.form["subject"],
      class_date=datetime.strptime(
          request.form["date"], "%Y-%m-%d"
      ).date(),
      status=request.form["status"]
  )

  db.session.add(record)
  db.session.commit()

  flash("Attendance saved.")

  if current_user.role=="student":

    rows = Attendance.query.filter_by(
        student_id=current_user.id
    ).all()

  else:

      rows = Attendance.query.order_by(
          Attendance.class_date.desc()
      ).limit(100).all()

      return render_template(
          "records.html",
          title="Attendance",
          rows=rows,
          kind="attendance"
      )

In [117]:
@app.route("/marks", methods=["GET", "POST"])
@login_required
def marks():

  if request.method == "POST":
    if current_user.role not in(
        "faculty", "main_admin","academic_admin"
    ):

      abort(403)

    record = Mark(
        student_id=int(request.form["student_id"]),
        subject=request.form["subject"],
        assessment=request.form["assessment"],
        score=float(request.form["score"]),
        maximum=float(request.form["maximum"])
    )

    if(
        record.maximum <=0
        or record.score <0
        or record.score> record.maximum
    ):

      abort(400)

    db.session.add(record)
    db.session.commit()

    flash("Mark saved.")

    if current_user.role == "student":

      rows = Mark.query.filter_by(
          student_id=current_user.id
      ).all()

    else:

        rows = Mark.query.order_by(
            Marks.id.desc()

        ).limit(100).all()

        return render_template(
            "records.html",
            title="Marks",
            rows=rows,
            kind="marks"
        )


In [122]:
@app.route("/announcements", methods=["GET", "POST"])
@login_required
def announcements():

  if request.method == "POST":

    if current_user.role not in(
        "main_admin","communication_admin", "faculty"
    ):

     abort(403)

    announcement = Announcement(
        title=request.form["title"],
        body=request.form["body"],
        author_id=current_user.id
    )

    db.session.add(announcement)
    db.session.commit()

    rows = Announcement.query.order_by(
        Announcement.created_at.desc()
    ).all()

    return render_template(
        "announcements.html",
        rows=rows
    )



In [124]:
@app.route("/events",methods=["GET","POST"])
@login_required
def events():

  if request.method == "POST":

    if current_user.role not in(
        "main_admin", "activity_admin", "faculty"
    ):

     abort(403)

     event = Event(
         title=request.form["title"],
         details=request.form["details"],
         event_date=datetime.strptime(
              request.form["date"], "%Y-%m-%d"
         ).date(),
         creator_id=current_user.id
     )

     db.session.add(event)
     db.session.commit()

     rows = Event.query.order_by(
         Event.event_date
     ).all()

     return render_template(
         "events.html",
         rows=rows
     )


In [131]:
@app.route("/admin/users", methods=["GET", "POST"])
@login_required
@allow("main_admin")
def users():

  if request.method == "POST":
    email = request.form["email"].lower().strip()

    if User.query.filter_by(email=email).first():
      flash("Email already exists.")
      all_users = User.query.all()
      return render_template("users.html", users=all_users)

    user = User(
        name=request.form["name"],
        email=email,
        department=request.form.get("department"),
        role=request.form["role"]
    )

    user.set_password(request.form["password"])
    db.session.add(user)
    db.session.commit()

  all_users = User.query.all()
  return render_template(
      "users.html",
      users=all_users
  )

In [132]:
@app.errorhandler(403)
def forbidden(error):

  return render_template(
      "message.html",
      message="You do not have permission to access this page."
  ),403

In [133]:
@app.cli.command("init-db")
def init_db():

  db.create_all()

  print("Database tables created successfully.")

In [134]:
@app.cli.command("create-admin")
def create_admin():

  email = input ("Admin email: ").lower().strip()

  if User.query.filter-by(email=email).first():
    print("Email already exists.")
    return

    name = input("Admin name:")
    password = input("Password(12+ characters):")

    if len(password)< 12:
      print("Password is too short.")
      return

      admin = User(
          name=name,
          email=email,
          role="main_admin"
      )

      admin.set_password(password)

      db.session.add(admin)
      db.session.commit()

      print("Main Admin created successfully.")

In [137]:
if __name__ == "__main__":
  app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
  File "/usr/local/lib/python3.13/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
  File "/usr/local/lib/python3.13/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information about how to avoid this problem.